# NIST PV fault labeling

Run [Data_Overview.ipynb](Data_Overview.ipynb) to merge the daily CSVs. Then select one site and year below, and run the cells in order. In Colab, the notebook reads merged data from `MyDrive/NIST_processed/` and exports results to `MyDrive/NIST_Labels/`. For local use, set `DATA_ROOT` to the directory containing `NIST_processed` and start from the repository root or this notebook's directory.

In [ ]:
import json
import os
import subprocess
import sys
from dataclasses import asdict
from pathlib import Path

import pandas as pd
from IPython.display import display

if "google.colab" in sys.modules:
    from google.colab import drive

    drive.mount("/content/drive")
    os.environ.setdefault("DATA_ROOT", "/content/drive/MyDrive")
    repo = Path("/content/pv-timeseries-fault-forecasting")
    if not repo.exists():
        subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "https://github.com/dulkusim/pv-timeseries-fault-forecasting.git",
                str(repo),
            ],
            check=True,
        )
else:
    cwd = Path.cwd()
    repo = next(
        (
            path
            for path in (cwd, *cwd.parents)
            if (path / "scripts" / "labeling.py").is_file()
        ),
        cwd,
    )

if not (repo / "scripts" / "qa.py").is_file():
    raise FileNotFoundError(
        f"Labeling and QA scripts must be available in {repo / 'scripts'}"
    )
sys.path.insert(0, str(repo))

from scripts import qa
from scripts.cleaning import missing_report
from scripts.config import CANOPY, GROUND
from scripts.feature_selection import (
    CORRELATION_COLUMNS,
    correlation_report,
    detect_faulty_rtds,
    rtd_report,
)
from scripts.inverter import ORIGINAL_INVERTER_POLICY as ORIGINAL_INVERTER_POLICY
from scripts.inverter import REVISED_INVERTER_POLICY
from scripts.io import get_data_root, load_site_year
from scripts.labeling import run_labeling, summarize
from scripts.reporting import label_distribution, report_stage
from scripts.strings import fit_string_baselines

## Select a site and year

The original inverter policy uses AC power below 0.05 kW for power-based shutdown, missing or nonpositive status for status-based shutdown, DC input above 1 kW for efficiency evaluation, and positive fault codes as reports. The revised inverter policy is the default. Select `ORIGINAL_INVERTER_POLICY` to use the historical inverter rules. After changing settings, rerun from this cell onward.

In [ ]:
SITE = "canopy"
YEAR = 2015
SAVE_CHECKPOINTS = False
OVERWRITE_FINAL = False
INVERTER_POLICY = REVISED_INVERTER_POLICY

In [ ]:
sites = {"canopy": CANOPY, "ground": GROUND}
if SITE not in sites:
    raise ValueError("SITE must be canopy or ground")
if type(YEAR) is not int or YEAR not in range(2015, 2019):
    raise ValueError("YEAR must be an integer from 2015 to 2018")

site = sites[SITE]
output_dir = get_data_root() / "NIST_Labels"
stem = f"{site.name}_{YEAR}"
records = []


def audit(name, frame, checkpoint=None):
    report_stage(
        name,
        frame,
        checkpoint,
        records=records,
        site=site,
        year=YEAR,
        save_checkpoints=SAVE_CHECKPOINTS,
    )

## Inspect the input and check coverage

The loader validates the timestamp year. QA rejects sentinel fill values before feature removal and reports missing minutes, duplicates and off-grid timestamps. Partial coverage is allowed. Rolling features use observation counts, so inspect gaps before interpreting those features.

In [ ]:
df = load_site_year(site, YEAR)
input_index = df.index.copy()
coverage = qa.check_year_coverage(df, YEAR)
qa.check_sentinels(df)

display(
    pd.Series(
        {
            key: value
            for key, value in coverage.items()
            if key not in {"missing_minutes", "off_grid_timestamps"}
        },
        name="coverage",
    )
)
if coverage["missing_minute_count"]:
    print("Missing minutes by month")
    display(pd.Series(coverage["missing_minutes"].month).value_counts().sort_index())
    print("First 20 missing minutes")
    display(coverage["missing_minutes"][:20])
if coverage["off_grid_count"]:
    display(coverage["off_grid_timestamps"][:20])

audit("Input", df)
print(f"DataFrame memory: {df.memory_usage(deep=True).sum() / 1024**2:,.1f} MiB")
display(df.head(), df.dtypes.to_frame("dtype"), df.describe().T)
display(df.groupby(df.index.month).size().rename("observed_rows"))

## Feature selection

Inspect annual RTD means and electrical correlations. The selected site's configuration determines RTD removal. The pipeline also removes secondary and redundant channels.

In [ ]:
display(rtd_report(df))
print("RTDs flagged by annual means:", detect_faulty_rtds(df))
correlation_columns = tuple(column for column in CORRELATION_COLUMNS if column in df)
display(correlation_report(df, columns=correlation_columns))
df = run_labeling(df, site, stages=("secondary", "redundant", "rtd"))
audit("Feature selection", df, "features")

## Cleaning and operating conditions

Review missingness before applying the shared column-removal policy and physical bounds. The operating mask identifies periods when the site's configured irradiance exceeds 100 W/m2.

In [ ]:
display(missing_report(df))
df = run_labeling(df, site, stages=("missing", "bounds"))
audit("Cleaning", df, "cleaned")
df = run_labeling(df, site, stages=("operating",))
audit("Operating mask", df, "operating")
display(df["Op_Mask"].value_counts(dropna=False))

## Performance ratio and power labels

Fuse plane-of-array irradiance and estimate module temperature. The pipeline calculates temperature-corrected expected power, weighted rolling performance ratios, and power-performance labels for the evaluable observations.

In [ ]:
df = run_labeling(
    df,
    site,
    stages=(
        "poa",
        "temperature",
        "pr_mask",
        "expected",
        "performance",
        "round",
        "power_labels",
    ),
)
audit("Performance ratio", df, "performance")
display(df[["POA_fused", "module_temp", "P_expected_AC_kW", "PR"]].describe().T)
display(label_distribution(df["label_power"]))

## String-level faults

Calculate string aggregates, voltage proxies and irradiance-normalized currents. Fit per-string baselines on evaluable observations from the selected year and use baseline-relative deviations to label string faults. For forecasting evaluation, fit the baselines on training data.

In [ ]:
df = run_labeling(df, site, stages=("aggregates", "voltage", "normalized"))
baselines = fit_string_baselines(df, site)
display(pd.Series(baselines, name="baseline").rename_axis("string"))
df = run_labeling(df, site, baselines=baselines, stages=("baselines", "string_labels"))
audit("String faults", df, "strings")
display(label_distribution(df["label_string"]))

## Inverter-level faults

Inspect inverter efficiency and DC discrepancy features, then compare shutdown and efficiency labels with fault codes reported by the inverter.

In [ ]:
df = run_labeling(df, site, inverter_policy=INVERTER_POLICY, stages=("inverter",))
audit("Inverter features", df, "inverter")
df = run_labeling(
    df, site, inverter_policy=INVERTER_POLICY, stages=("inverter_labels",)
)
audit("Inverter labels", df)
display(label_distribution(df["label_inverter"]))
display(qa.cross_check_with_inverter_flags(df))

## Validate and export

Verify that labeling preserved every timestamp, then review counts and the power-versus-inverter label table. The export cell saves the labeled CSV, label summary, stage audit and methodology. Enable `OVERWRITE_FINAL` to replace existing results. Optional checkpoints reject existing files.

In [ ]:
if not df.index.equals(input_index):
    raise ValueError("Labeling changed the input timestamps")
summary = summarize(df)
display(summary)
display(pd.crosstab(df["label_power"], df["label_inverter"]))
audit("Validated labels", df, "labeled")

In [ ]:
paths = {
    "labels": output_dir / f"{stem}.csv",
    "summary": output_dir / f"{stem}_label_summary.csv",
    "audit": output_dir / f"{stem}_stage_audit.csv",
    "methodology": output_dir / f"{stem}_methodology.json",
}
existing = [str(path) for path in paths.values() if path.exists()]
if existing and not OVERWRITE_FINAL:
    raise FileExistsError("Set OVERWRITE_FINAL to replace: " + ", ".join(existing))

commit = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=repo, text=True
).strip()
methodology = {
    "site": asdict(site),
    "year": YEAR,
    "inverter_policy": asdict(INVERTER_POLICY),
    "string_baselines": baselines,
    "repository_commit": commit,
}
output_dir.mkdir(parents=True, exist_ok=True)
mode = "w" if OVERWRITE_FINAL else "x"
df.to_csv(paths["labels"], mode=mode, index_label="TIMESTAMP")
summary.to_csv(paths["summary"], mode=mode, index=False)
pd.DataFrame(records).to_csv(paths["audit"], mode=mode, index=False)
with paths["methodology"].open(mode, encoding="utf-8") as handle:
    json.dump(methodology, handle, indent=2)
    handle.write("\n")
for path in paths.values():
    print(path)